In [2]:
import pandas as pd
import duckdb
data = [
    # Device A：有连续 ERROR、月环比、增长率、累计分析
    ["A", "2026-01-01", "NORMAL", 2],
    ["A", "2026-01-02", "ERROR", 5],
    ["A", "2026-01-03", "ERROR", 6],
    ["A", "2026-01-04", "ERROR", 4],
    ["A", "2026-01-05", "NORMAL", 1],
    ["A", "2026-02-01", "NORMAL", 3],
    ["A", "2026-02-02", "ERROR", 8],
    ["A", "2026-02-03", "ERROR", 7],
    ["A", "2026-02-04", "NORMAL", 2],

    # Device B：有短 ERROR、连续 ERROR、缺少上月同日
    ["B", "2026-01-01", "NORMAL", 1],
    ["B", "2026-01-02", "ERROR", 4],
    ["B", "2026-01-03", "NORMAL", 2],
    ["B", "2026-01-04", "ERROR", 6],
    ["B", "2026-01-05", "ERROR", 5],
    ["B", "2026-02-01", "ERROR", 3],
    ["B", "2026-02-03", "NORMAL", 2],
    ["B", "2026-02-04", "ERROR", 9],
    ["B", "2026-02-05", "ERROR", 10],

    # Device C：包含 previous value = 0 的增长率场景
    ["C", "2026-01-01", "NORMAL", 0],
    ["C", "2026-01-02", "NORMAL", 0],
    ["C", "2026-01-03", "ERROR", 3],
    ["C", "2026-01-04", "ERROR", 2],
    ["C", "2026-02-01", "NORMAL", 5],
    ["C", "2026-02-02", "ERROR", 4],
    ["C", "2026-02-03", "ERROR", 6],
    ["C", "2026-02-04", "ERROR", 7],

    # Device D：只有后期数据，部分没有历史基准
    ["D", "2026-02-01", "NORMAL", 2],
    ["D", "2026-02-02", "ERROR", 5],
    ["D", "2026-02-03", "NORMAL", 1],
    ["D", "2026-02-04", "ERROR", 6],
    ["D", "2026-02-05", "ERROR", 8],
]

df = pd.DataFrame(
    data,
    columns=["device_id", "stat_date", "status", "alarm_count"]
)

df["stat_date"] = pd.to_datetime(df["stat_date"])

df = df.sort_values(["device_id", "stat_date"]).reset_index(drop=True)

print(df)



   device_id  stat_date  status  alarm_count
0          A 2026-01-01  NORMAL            2
1          A 2026-01-02   ERROR            5
2          A 2026-01-03   ERROR            6
3          A 2026-01-04   ERROR            4
4          A 2026-01-05  NORMAL            1
5          A 2026-02-01  NORMAL            3
6          A 2026-02-02   ERROR            8
7          A 2026-02-03   ERROR            7
8          A 2026-02-04  NORMAL            2
9          B 2026-01-01  NORMAL            1
10         B 2026-01-02   ERROR            4
11         B 2026-01-03  NORMAL            2
12         B 2026-01-04   ERROR            6
13         B 2026-01-05   ERROR            5
14         B 2026-02-01   ERROR            3
15         B 2026-02-03  NORMAL            2
16         B 2026-02-04   ERROR            9
17         B 2026-02-05   ERROR           10
18         C 2026-01-01  NORMAL            0
19         C 2026-01-02  NORMAL            0
20         C 2026-01-03   ERROR            3
21        

# 设备每日状态与报警综合分析

## 一、题目背景

你现在有一张设备每日运行状态表：

| 字段名 | 含义 |
|---|---|
| device_id | 设备 ID |
| stat_date | 统计日期 |
| status | 当天设备状态 |
| alarm_count | 当天报警次数 |

现在要从这张表里完成一组综合分析，判断设备异常趋势、连续异常区间、最新异常记录、累计报警情况，以及月度对比变化。

---

## 二、任务要求

### Task 1：状态变化检测

判断每个设备当天状态是否相比上一条记录发生变化。

输出字段：

```text
device_id
stat_date
status
previous_status
is_status_changed
```

要求：

```text
每个设备第一条记录没有上一条状态，不算状态变化。
```

---

### Task 2：连续 ERROR 区间识别

找出每个设备中，连续处于 `ERROR` 状态达到 2 天及以上的区间。

输出字段：

```text
device_id
error_start_date
error_end_date
error_days
```

要求：

```text
只统计连续 ERROR 区间。
连续天数 >= 2 才输出。
```

---

### Task 3：每个设备最近一次 ERROR 记录

找出每个设备最近一次 `ERROR` 状态记录。

输出字段：

```text
device_id
stat_date
status
alarm_count
```

要求：

```text
没有 ERROR 记录的设备不输出。
如果同一设备同一天只有一条记录，直接取最近日期。
```

---

### Task 4：每个设备报警次数最高的前 2 天

找出每个设备 `alarm_count` 最高的前 2 天。

输出字段：

```text
device_id
stat_date
status
alarm_count
rn
```

要求：

```text
每个设备最多输出 2 条。
如果 alarm_count 相同，stat_date 较晚的排前面。
```

---

### Task 5：累计报警次数

计算每个设备截至当天的累计报警次数。

输出字段：

```text
device_id
stat_date
status
alarm_count
running_alarm_count
```

要求：

```text
每个设备内部按 stat_date 从早到晚累计。
```

---

### Task 6：最近 2 条记录的平均报警次数

计算每个设备当前记录和上一条记录的平均报警次数。

输出字段：

```text
device_id
stat_date
status
alarm_count
moving_avg_2_alarm_count
```

要求：

```text
每个设备第一条记录只有自己一条，也要计算平均值。
```

---

### Task 7：上个月同日报警次数对比

计算每个设备当天报警次数与上个月同日相比的变化量和增长率。

输出字段：

```text
device_id
stat_date
status
alarm_count
previous_month_alarm_count
alarm_count_diff
growth_rate
growth_rate_pct
```

要求：

```text
如果没有上个月同日记录：
previous_month_alarm_count = NULL / NaN
alarm_count_diff = NULL / NaN
growth_rate = NULL / NaN
growth_rate_pct = NULL / NaN

如果上个月同日报警次数为 0：
alarm_count_diff 可以正常计算
growth_rate = NULL / NaN
growth_rate_pct = NULL / NaN
```

---

## 三、建议 Notebook 结构

```text
01 数据准备
02 Task 1：状态变化检测
03 Task 2：连续 ERROR 区间
04 Task 3：最近一次 ERROR
05 Task 4：报警次数 Top 2
06 Task 5：累计报警次数
07 Task 6：最近 2 条移动平均
08 Task 7：上个月同日对比与增长率
09 总结：四类 Pattern 的使用场景
```

In [4]:
# ========================
# Task1(SQL轨道)
# ========================

query = """
WITH previous_status_table AS (

    SELECT
        device_id,
        stat_date,
        status,
        alarm_count,
        LAG(status)
            OVER(
                PARTITION BY device_id 
                ORDER BY stat_date
            ) AS previous_status
    FROM df
)
SELECT
    device_id,
    stat_date,
    status,
    alarm_count,
    previous_status,
    CASE
        WHEN status != previous_status 
        AND  previous_status IS NOT NULL
        THEN True
        ELSE False
    END AS is_status_changed
FROM previous_status_table
ORDER BY device_id,stat_date;
"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,device_id,stat_date,status,alarm_count,previous_status,is_status_changed
0,A,2026-01-01,NORMAL,2,NaN,False
1,A,2026-01-02,ERROR,5,NORMAL,True
2,A,2026-01-03,ERROR,6,ERROR,False
3,A,2026-01-04,ERROR,4,ERROR,False
4,A,2026-01-05,NORMAL,1,ERROR,True
5,A,2026-02-01,NORMAL,3,NORMAL,False
6,A,2026-02-02,ERROR,8,NORMAL,True
7,A,2026-02-03,ERROR,7,ERROR,False
8,A,2026-02-04,NORMAL,2,ERROR,True
9,B,2026-01-01,NORMAL,1,NaN,False


In [3]:
# ========================
# Task1(PANDAS轨道)
# ========================

df_pd  = (
    df
    .sort_values(by=['device_id','stat_date'])
    .assign(
        previous_status = lambda x:(
            x.groupby('device_id')['status']
            .shift(1)
        ),
        is_status_changed = lambda x:(
            x['previous_status'].notna() & (x['status'] != x['previous_status'])
        )
    )
    .reset_index(drop=True)
)
df_pd

,device_id,stat_date,status,alarm_count,previous_status,is_status_changed
0,A,2026-01-01,NORMAL,2,NaN,False
1,A,2026-01-02,ERROR,5,NORMAL,True
2,A,2026-01-03,ERROR,6,ERROR,False
3,A,2026-01-04,ERROR,4,ERROR,False
4,A,2026-01-05,NORMAL,1,ERROR,True
5,A,2026-02-01,NORMAL,3,NORMAL,False
6,A,2026-02-02,ERROR,8,NORMAL,True
7,A,2026-02-03,ERROR,7,ERROR,False
8,A,2026-02-04,NORMAL,2,ERROR,True
9,B,2026-01-01,NORMAL,1,NaN,False


In [44]:

query2 = """
WITH status_bool AS (
    SELECT
        device_id,
        stat_date,
        status,
        status = 'ERROR' AS status_is_error
    FROM df
),

previous_status AS (
    SELECT
        device_id,
        stat_date,
        status,
        status_is_error,
        COALESCE(
            LAG(status_is_error) OVER (
                PARTITION BY device_id
                ORDER BY stat_date
            ),
            FALSE
        ) AS previous_is_error
    FROM status_bool
),

start_sign_table AS (
    SELECT
        device_id,
        stat_date,
        status,
        status_is_error,
        CASE
            WHEN status_is_error = TRUE
                 AND previous_is_error = FALSE
            THEN 1
            ELSE 0
        END AS error_start_sign
    FROM previous_status
),

phase_sign_table AS (
    SELECT
        device_id,
        stat_date,
        status,
        status_is_error,
        SUM(error_start_sign) OVER (
            PARTITION BY device_id
            ORDER BY stat_date
        ) AS phase_sign
    FROM start_sign_table
),

final_table AS (
    SELECT
        device_id,
        phase_sign,
        MIN(stat_date) AS error_start_date,
        MAX(stat_date) AS error_end_date,
        COUNT(*) AS error_days
    FROM phase_sign_table
    WHERE status_is_error = TRUE
    GROUP BY device_id, phase_sign
)

SELECT
    device_id,
    error_start_date,
    error_end_date,
    error_days
FROM final_table
WHERE error_days >= 2
ORDER BY device_id, error_start_date;
"""
df_sql = duckdb.execute(query2).fetchdf()
df_sql

,device_id,error_start_date,error_end_date,error_days
0,A,2026-01-02,2026-01-04,3
1,A,2026-02-02,2026-02-03,2
2,B,2026-01-04,2026-02-01,3
3,B,2026-02-04,2026-02-05,2
4,C,2026-01-03,2026-01-04,2
5,C,2026-02-02,2026-02-04,3
6,D,2026-02-04,2026-02-05,2


In [43]:
# ========================
# Task2（Pandas 轨道）
# ========================

df_task2 = (
    df
    .sort_values(by=['device_id', 'stat_date'])
    .copy()
)

df_task2['is_error'] = df_task2['status'].eq('ERROR')

df_task2['previous_is_error'] = (
    df_task2
    .groupby('device_id')['is_error']
    .shift(1, fill_value=False)
)

df_task2['error_start'] = (
    df_task2['is_error']
    & ~df_task2['previous_is_error']
).astype(int)

df_task2['phase_sign'] = (
    df_task2
    .groupby('device_id')['error_start']
    .cumsum()
)

df_pd = (
    df_task2
    .loc[df_task2['is_error']]
    .groupby(
        ['device_id', 'phase_sign'],
        as_index=False
    )
    .agg(
        error_start_date=('stat_date', 'min'),
        error_end_date=('stat_date', 'max'),
        error_days=('stat_date', 'size')
    )
    .query('error_days >= 2')
    [
        [
            'device_id',
            'error_start_date',
            'error_end_date',
            'error_days'
        ]
    ]
    .sort_values(
        ['device_id', 'error_start_date']
    )
    .reset_index(drop=True)
)
df_pd

,device_id,error_start_date,error_end_date,error_days
0,A,2026-01-02,2026-01-04,3
1,A,2026-02-02,2026-02-03,2
2,B,2026-01-04,2026-02-01,3
3,B,2026-02-04,2026-02-05,2
4,C,2026-01-03,2026-01-04,2
5,C,2026-02-02,2026-02-04,3
6,D,2026-02-04,2026-02-05,2
